## Data Analysis

**【Data Science Project】 Building a Machine Learning Pipeline for a Predictive Car Price Model with PySpark**

In [63]:
import os

# point java home to actual conda package reference
os.environ["JAVA_HOME"] = "/Users/andreasliistro/mambaforge/pkgs/openjdk-22.0.1-hbeb2e11_0/lib/jvm"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import isnan, when, count, col, lit, lower, trim, regexp_replace
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder

import re

# init spark session
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [64]:
# load data into spark
data = spark.read.csv("ML-pipeLine-car-prediction/data/vehicles.csv", header=True, inferSchema=True, multiLine=True)
data.show()

+----------+--------------------+--------------------+--------------------+-----+----+------------+-----+---------+---------+----+--------+------------+------------+----+-----+----+----+-----------+---------+-----------+------+-----+----+----+------------+
|        id|                 url|              region|          region_url|price|year|manufacturer|model|condition|cylinders|fuel|odometer|title_status|transmission| VIN|drive|size|type|paint_color|image_url|description|county|state| lat|long|posting_date|
+----------+--------------------+--------------------+--------------------+-----+----+------------+-----+---------+---------+----+--------+------------+------------+----+-----+----+----+-----------+---------+-----------+------+-----+----+----+------------+
|7222695916|https://prescott....|            prescott|https://prescott....| 6000|NULL|        NULL| NULL|     NULL|     NULL|NULL|    NULL|        NULL|        NULL|NULL| NULL|NULL|NULL|       NULL|     NULL|       NULL|  NULL|  

In [65]:
# show schema
data.printSchema()

root
 |-- id: long (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: integer (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- posting_date: string (null

In [66]:
# show statisctics
data.describe().toPandas().transpose()

,0,1,2,3,4
summary,count,mean,stddev,min,max
id,426582,7.311485308380555E9,4473540.669232186,7207408119,7317101084
url,426582,None,None,https://abilene.craigslist.org/ctd/d/abilene-2...,https://zanesville.craigslist.org/cto/d/zanesv...
region,426582,None,None,SF bay area,zanesville / cambridge
region_url,426582,None,None,https://abilene.craigslist.org,https://zanesville.craigslist.org
price,426582,75240.00097753773,1.218653645968038E7,0,3736928711
year,425377,2011.2356544900172,9.4529441236719,1900,2022
manufacturer,408948,None,None,acura,volvo
model,421310,1955.136086248983,5437.026017668185,"""""""t""""""",🔥GMC Sierra 1500 SLE🔥 4X4 🔥
condition,252581,None,None,excellent,salvage


In [67]:
# drop columns not interested in
data = data.drop("url",
                 "region_url", 
                 "image_url", 
                 "vin", 
                 "lat", 
                 "long",
                 "region",
                 "description"
                 )

# handle "description" differently


In [68]:
# analyse missing values
data.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in data.columns]).show()

# analyis missing values in percentage
data.select([(count(when(isnan(c) | col(c).isNull(), c)) / count(lit(1)) * 100).alias(c) for c in data.columns]).show()


+---+-----+----+------------+-----+---------+---------+----+--------+------------+------------+------+------+-----+-----------+------+-----+------------+
| id|price|year|manufacturer|model|condition|cylinders|fuel|odometer|title_status|transmission| drive|  size| type|paint_color|county|state|posting_date|
+---+-----+----+------------+-----+---------+---------+----+--------+------------+------------+------+------+-----+-----------+------+-----+------------+
|  0|    0|1205|       17634| 5272|   174001|   177571|3013|    4400|        8237|        2556|130475|306182|92787|     130112|367950| 2535|        1773|
+---+-----+----+------------+-----+---------+---------+----+--------+------------+------------+------+------+-----+-----------+------+-----+------------+



+---+-----+-------------------+-----------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+----------------+------------------+-------------------+
| id|price|               year|     manufacturer|             model|         condition|        cylinders|             fuel|          odometer|      title_status|      transmission|             drive|             size|             type|      paint_color|          county|             state|       posting_date|
+---+-----+-------------------+-----------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+----------------+------------------+-------------------+
|0.0|  0.0|0.28247792921407844|4.133789048764364|1.2358702430013457|40

Checking empty / null values shows following columns with more than 1/3 empty values, which can be removed
- cylinders
- county

Following additional columns contain high number of empty values but identified as important columns, therefore can't be removed
- paint_color
- condition
- drive
- size

In [69]:
# delete columns with larger number of missing values
data = data.drop("cylinders", "county")


In [70]:
def print_data_size(data, column, before=False):
    keyword = "before" if before else "after"
    print(f"Data size {keyword} filtering {column}: {data.count()}")

In [71]:
# delete rows with missing price
data = data.na.drop(subset=["price"])

# convert "price" to double and remove invalid doubles values
data = data.withColumn("price", data["price"].cast("double"))

print_data_size(data, "price")

Data size after filtering price: 426582


In [72]:
# leave nulls in but replace unreal values with null
data = data.withColumn("year", when(col("year") < 1900, None).otherwise(col("year")))
data = data.withColumn("year", when(col("year") > 2025, None).otherwise(col("year")))

# convert "year" to double
data = data.withColumn("year", data["year"].cast("double"))

print_data_size(data, "year")

Data size after filtering year: 426582


In [73]:
# convert values in manufacturer to lower case and trim whitespaces
data = data.withColumn("manufacturer", col("manufacturer").cast("string"))
data = data.withColumn("manufacturer", lower(col("manufacturer")))
data = data.withColumn("manufacturer", trim(col("manufacturer")))

# replace nissal silverado with nissan
data = data.withColumn("manufacturer", when(col("manufacturer") == "nissansilverado", "nissan").otherwise(col("manufacturer")))

manufacture_counts = data.groupBy("manufacturer").count().orderBy(col("count").desc())
manufacture_counts.show(1000, False)

print_data_size(data, "manufacturer")

+---------------+-----+
|manufacturer   |count|
+---------------+-----+
|ford           |70934|
|chevrolet      |55023|
|toyota         |34177|
|honda          |21257|
|nissan         |19055|
|jeep           |19003|
|ram            |18327|
|NULL           |17634|
|gmc            |16760|
|bmw            |14684|
|dodge          |13703|
|mercedes-benz  |11808|
|hyundai        |10335|
|subaru         |9494 |
|volkswagen     |9342 |
|kia            |8452 |
|lexus          |8190 |
|audi           |7564 |
|cadillac       |6950 |
|chrysler       |6026 |
|acura          |5977 |
|buick          |5491 |
|mazda          |5423 |
|infiniti       |4798 |
|lincoln        |4218 |
|volvo          |3374 |
|mitsubishi     |3292 |
|mini           |2375 |
|pontiac        |2288 |
|rover          |2113 |
|jaguar         |1945 |
|porsche        |1383 |
|mercury        |1184 |
|saturn         |1089 |
|alfa-romeo     |897  |
|tesla          |866  |
|fiat           |792  |
|harley-davidson|153  |
|ferrari        

Data size after filtering manufacturer: 426582


In [74]:
# convert values in manufacturer to lower case and trim whitespaces
data = data.withColumn("model", col("model").cast("string"))
data = data.withColumn("model", lower(col("model")))
data = data.withColumn("model", trim(col("model")))

data = data.withColumn("model", regexp_replace(col("model"), r'[^A-Za-z0-9\s/-]', ''))

# feature filtering for model
model_counts = data.groupBy("model").count().orderBy(col("count").desc())

low_number_models = model_counts.filter(col("count") < 3)
print("Number of models with low count:", low_number_models.count())

# remove models from data with low count
data = data.join(low_number_models, "model", "left_anti")

print_data_size(data, "model")

Number of models with low count: 18063


Data size after filtering model: 404770


In [75]:
# clean condition data
condition_counts = data.groupBy("condition").count().orderBy(col("count").desc())
condition_counts.show(1000, False)

print_data_size(data, "condition")

+---------+------+
|condition|count |
+---------+------+
|NULL     |166151|
|good     |116890|
|excellent|94629 |
|like new |19374 |
|fair     |6057  |
|new      |1122  |
|salvage  |547   |
+---------+------+



Data size after filtering condition: 404770


In [76]:
# clean fuel data
fuel_counts = data.groupBy("fuel").count().orderBy(col("count").desc())
fuel_counts.show(1000, False)

print_data_size(data, "fuel")

+--------+------+
|fuel    |count |
+--------+------+
|gas     |338507|
|other   |29805 |
|diesel  |27681 |
|hybrid  |4902  |
|NULL    |2335  |
|electric|1540  |
+--------+------+



Data size after filtering fuel: 404770


In [77]:
# odometer cleaning
data = data.withColumn("odometer", when(col("odometer") < 0, None).otherwise(col("odometer")))

# convert "odometer" to double
data = data.withColumn("odometer", data["odometer"].cast("double"))

print_data_size(data, "odometer")

Data size after filtering odometer: 404770


In [78]:
# transmission cleaning
transmission_counts = data.groupBy("transmission").count().orderBy(col("count").desc())
transmission_counts.show(1000, False)

print_data_size(data, "transmission")

# supported_transmission = [
#   "automatic",
#   "manual",
#   "other",
#   "unknown",
# ]

# # replace Null with unknown
# data = data.fillna("unknown", subset=["transmission"])

# # replace all other values with other
# data = data.withColumn("transmission", when(col("transmission").isin(supported_transmission), col("transmission")).otherwise("other"))

# # show new values for transmission
# data.groupBy("transmission").count().orderBy(col("count").desc()).show(20)

+------------+------+
|transmission|count |
+------------+------+
|automatic   |318127|
|other       |61829 |
|manual      |22339 |
|NULL        |2475  |
+------------+------+



Data size after filtering transmission: 404770


In [79]:
# # drive cleaning
# drive_counts = data.groupBy("drive").count().orderBy(col("count").desc())

# supported_drive = [
#   "fwd",
#   "4wd",
#   "rwd",
#   "unknown",
# ]

# # replace Null with unknown
# data = data.fillna("unknown", subset=["drive"])

# # replace all other values with other
# data = data.withColumn("drive", when(col("drive").isin(supported_drive), col("drive")).otherwise("other"))

# # show new values for drive
# data.groupBy("drive").count().orderBy(col("count").desc()).show(20)

In [80]:
# size cleaning
# size_counts = data.groupBy("size").count().orderBy(col("count").desc())

# supported_size = [
#   "full-size",
#   "mid-size",
#   "compact",
#   "sub-compact",
#   "unknown",
# ]

# # replace Null with unknown
# data = data.fillna("unknown", subset=["size"])

# # replace all other values with other
# data = data.withColumn("size", when(col("size").isin(supported_size), col("size")).otherwise("other"))

# # show new values for size
# data.groupBy("size").count().orderBy(col("count").desc()).show(20)

In [81]:
# state cleaning
# state_counts = data.groupBy("state").count().orderBy(col("count").desc())

# # list of all US states shortcodes
# supported_states = [
#   "al", "ak", "az", "ar", "ca", "co", "ct", "de", "fl", "ga", "hi", "id", "il", "in", "ia", "ks", "ky", "la", "me", "md", "ma", "mi", "mn", "ms", "mo", "mt", "ne", "nv", "nh", "nj", "nm", "ny", "nc", "nd", "oh", "ok", "or", "pa", "ri", "sc", "sd", "tn", "tx", "ut", "vt", "va", "wa", "wv", "wi", "wy"
# ]

# # replace Null with unknown
# data = data.fillna("unknown", subset=["state"])

# # replace all other values with other
# data = data.withColumn("state", when(col("state").isin(supported_states), col("state")).otherwise("other"))

# # show new values for state
# data.groupBy("state").count().orderBy(col("count").desc()).show(20)

In [82]:
# paint_color cleaning
# paint_color_counts = data.groupBy("paint_color").count().orderBy(col("count").desc())

# supported_paint_color = [
#   "white",
#   "black",
#   "silver",
#   "blue",
#   "red",
#   "grey",
#   "green",
#   "brown",
#   "yellow",
#   "custom",
#   "orange",
#   "purple",
#   "unknown",
# ]

# # replace Null with unknown
# data = data.fillna("unknown", subset=["paint_color"])

# # replace all other values with other
# data = data.withColumn("paint_color", when(col("paint_color").isin(supported_paint_color), col("paint_color")).otherwise("other"))

# # show new values for paint_color
# data.groupBy("paint_color").count().orderBy(col("count").desc()).show(20)

In [83]:
# show new schema
data.printSchema()

root
 |-- model: string (nullable = true)
 |-- id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- year: double (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- state: string (nullable = true)
 |-- posting_date: string (nullable = true)



In [84]:
# show statistics
data.describe().toPandas().transpose()

,0,1,2,3,4
summary,count,mean,stddev,min,max
model,399498,1769.0351324666776,1209.0341811193025,,zl1 camaro
id,404770,7.311441030127712E9,4479145.211212143,7207408119,7317098055
price,404770,77149.20430614917,1.2504261601552498E7,0.0,3.736928711E9
year,403648,2011.5387614951642,8.892779822265082,1900.0,2022.0
manufacturer,391786,None,None,acura,volvo
condition,238619,None,None,excellent,salvage
fuel,402435,None,None,diesel,other
odometer,400631,96590.75641176045,194646.41772511773,0.0,1.0E7
title_status,397581,None,None,clean,salvage


In [89]:
data_export = data.select(
  "id",
  "price",
  "year",
  "manufacturer",
  "model",
  "condition",
  "fuel",
  "odometer",
  "transmission",
)

data_export.show()

print("Exporting cleaned data to csv, count: ", data_export.count())

# safe data as new csv
data_export.write.csv("ML-pipeLine-car-prediction/data/vehicles_cleaned.csv", header=True)

+----------+-------+------+------------+---------+---------+----+--------+------------+
|        id|  price|  year|manufacturer|    model|condition|fuel|odometer|transmission|
+----------+-------+------+------------+---------+---------+----+--------+------------+
|7316535686| 3399.0|2006.0|       buick| lacrosse|     NULL| gas|160218.0|   automatic|
|7315477708| 4400.0|2006.0|       buick| lacrosse|     NULL| gas|124201.0|   automatic|
|7315258080|17977.0|2012.0|      toyota|  4runner|     NULL| gas|182263.0|   automatic|
|7314505215| 3399.0|2006.0|       buick| lacrosse|     NULL| gas|160218.0|   automatic|
|7312181319| 4400.0|2006.0|       buick| lacrosse|     NULL| gas|124201.0|   automatic|
|7310532358| 3399.0|2006.0|       buick| lacrosse|     NULL| gas|160218.0|   automatic|
|7309379404| 4400.0|2006.0|       buick| lacrosse|     NULL| gas|124201.0|   automatic|
|7307956167| 3399.0|2006.0|       buick| lacrosse|     NULL| gas|160218.0|   automatic|
|7307662967|18477.0|2011.0|     

Exporting cleaned data to csv, count:  404770


25/01/02 16:04:35 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 965804 ms exceeds timeout 120000 ms
25/01/02 16:04:35 WARN SparkContext: Killing executors is not supported by current scheduler.
25/01/02 16:04:37 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$